In [1]:
import pandas as pd

In [2]:
reviews=pd.read_csv("C:/Users/Supradha/ E-Commerce User Funnel & Drop-off Analysis/dataset/olist_order_reviews_dataset.csv")

In [3]:
reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [4]:
reviews.shape

(99224, 7)

In [5]:
reviews.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [6]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [7]:
reviews.duplicated().sum()

0

In [8]:
reviews['review_id'].duplicated().sum()

814

In [9]:
# Handle duplicate review_ids
# One order can have multiple review submissions — keep latest
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')

In [10]:
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

In [11]:
reviews.sort_values('review_answer_timestamp',ascending=False, inplace=True)

In [12]:
reviews.drop_duplicates(subset='order_id', keep='first', inplace=True)

In [13]:
reviews.shape

(98673, 7)

In [14]:
# Validate review_score 
print(reviews['review_score'].value_counts().sort_index())

1    11363
2     3131
3     8133
4    19038
5    57008
Name: review_score, dtype: int64


In [15]:
invalid_scores = reviews[~reviews['review_score'].isin([1, 2, 3, 4, 5])]

In [16]:
print(f"\n  Invalid scores (not 1-5) : {len(invalid_scores)}")
reviews = reviews[reviews['review_score'].isin([1, 2, 3, 4, 5])]


  Invalid scores (not 1-5) : 0


In [17]:
# ── FIX: Enforce correct dtype on review_score ────────────────
reviews['review_score'] = pd.to_numeric(
    reviews['review_score'], errors='coerce'
).astype('Int64')

In [18]:
# Clean text columns
if 'review_comment_title' in reviews.columns:
    reviews['review_comment_title'] = (reviews['review_comment_title'].str.strip().fillna(''))

In [19]:
if 'review_comment_message' in reviews.columns:
    reviews['review_comment_message'] = (reviews['review_comment_message'].str.strip().fillna(''))

In [20]:
# ── 4.4  Flag reviews with no comment ────────────────────────
reviews['has_comment'] = (reviews['review_comment_message'].str.len() > 0)

In [21]:
print(f"\nReviews with comment    : {reviews['has_comment'].sum()}")


Reviews with comment    : 40748


In [22]:
print(f"Reviews without comment : {(~reviews['has_comment']).sum()}") 

Reviews without comment : 57925


In [23]:
# Sentiment bucket
reviews['sentiment'] = pd.cut(
    reviews['review_score'],
    bins=[0, 2, 3, 5],
    labels=['Negative', 'Neutral', 'Positive'])

In [24]:
reviews['sentiment'].value_counts()

Positive    76046
Negative    14494
Neutral      8133
Name: sentiment, dtype: int64

In [25]:
reviews.shape

(98673, 9)

In [26]:
reviews.to_csv("cleaned_reviews.csv",index=False)